<a href="https://colab.research.google.com/github/00ayeshazahra/WWTP-Predictive-Maintainance/blob/main/shutdown_prediction_melbourne.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import os
print('All libraries loaded successfully.')

All libraries loaded successfully.


In [ ]:
import kagglehub
import os

# Download the Full-Scale Waste Water Treatment Plant (Melbourne) dataset from Kaggle
path = kagglehub.dataset_download('d4rklucif3r/full-scale-waste-water-treatment-plant-data')
print('Dataset downloaded to:', path)

DATA_PATH = os.path.join(path, 'Data-Melbourne_F_fixed.csv')

df = pd.read_csv(DATA_PATH, index_col='Unnamed: 0')

# Convert Energy Consumption from Wh to MWh
df['Energy Consumption'] = round(df['Energy Consumption'] / 1000, 1)

print(f'Dataset shape: {df.shape}')
df.head()

Using Colab cache for faster access to the 'full-scale-waste-water-treatment-plant-data' dataset.
Dataset downloaded to: /kaggle/input/full-scale-waste-water-treatment-plant-data
Dataset shape: (1382, 19)


,Average Outflow,Average Inflow,Energy Consumption,Ammonia,Biological Oxygen Demand,Chemical Oxygen Demand,Total Nitrogen,Average Temperature,Maximum temperature,Minimum temperature,Atmospheric pressure,Average humidity,Total rainfall,Average visibility,Average wind speed,Maximum wind speed,Year,Month,Day
0,2.941,2.589,175.9,27.0,365.0,730.0,60.378,19.3,25.1,12.6,0.0,56.0,1.52,10.0,26.9,53.5,2014.0,1.0,1.0
1,2.936,2.961,181.6,25.0,370.0,740.0,60.026,17.1,23.6,12.3,0.0,63.0,0.00,10.0,14.4,27.8,2014.0,1.0,2.0
2,2.928,3.225,202.0,42.0,418.0,836.0,64.522,16.8,27.2,8.8,0.0,47.0,0.25,10.0,31.9,61.1,2014.0,1.0,5.0
3,2.928,3.354,207.5,36.0,430.0,850.0,63.000,14.6,19.9,11.1,0.0,49.0,0.00,10.0,27.0,38.9,2014.0,1.0,6.0
4,2.917,3.794,202.8,46.0,508.0,1016.0,65.590,13.4,19.1,8.0,0.0,65.0,0.00,10.0,20.6,35.2,2014.0,1.0,7.0


In [ ]:
# Column groups from the LuciferML notebook
atmospheric_cols = [
    'Average Temperature', 'Maximum temperature', 'Minimum temperature',
    'Atmospheric pressure', 'Average humidity', 'Total rainfall',
    'Average visibility', 'Average wind speed', 'Maximum wind speed',
]
time_cols = ['Year', 'Month', 'Day']
process_cols = [
    'Chemical Oxygen Demand', 'Biological Oxygen Demand',
    'Ammonia', 'Total Nitrogen', 'Average Inflow', 'Average Outflow',
    'Energy Consumption'
]

print('Missing values:')
print(df.isnull().sum())
print()
df.describe().T

Missing values:
Average Outflow             0
Average Inflow              0
Energy Consumption          0
Ammonia                     0
Biological Oxygen Demand    0
Chemical Oxygen Demand      0
Total Nitrogen              0
Average Temperature         0
Maximum temperature         0
Minimum temperature         0
Atmospheric pressure        0
Average humidity            0
Total rainfall              0
Average visibility          0
Average wind speed          0
Maximum wind speed          0
Year                        0
Month                       0
Day                         0
dtype: int64



,count,mean,std,min,25%,50%,75%,max
Average Outflow,1382.0,3.930608,1.228778,0.000004,3.07450,3.7010,4.49875,7.920
Average Inflow,1382.0,4.506338,1.439583,2.589000,3.64325,4.1615,4.84775,18.968
Energy Consumption,1382.0,275.159479,44.639574,116.600000,246.42500,275.8000,305.67500,398.300
Ammonia,1382.0,39.222302,7.761598,13.000000,34.00000,39.0000,44.00000,93.000
Biological Oxygen Demand,1382.0,382.061708,85.996012,140.000000,330.00000,360.0000,422.98000,850.000
Chemical Oxygen Demand,1382.0,845.960434,145.416540,360.000000,751.25000,845.0000,920.00000,1700.000
Total Nitrogen,1382.0,62.740752,3.571035,40.000000,61.39600,62.9575,64.36600,92.000
Average Temperature,1382.0,15.036686,5.398491,0.000000,10.80000,14.3000,18.57500,35.500
Maximum temperature,1382.0,20.530897,7.096760,0.000000,15.00000,19.2000,25.20000,43.500
Minimum temperature,1382.0,10.037337,4.656887,-2.000000,6.80000,9.6000,13.00000,28.500


In [ ]:
#Stress scale
scaler_tmp = MinMaxScaler()
stress_features = ['Chemical Oxygen Demand', 'Biological Oxygen Demand',
                   'Ammonia', 'Total Nitrogen', 'Energy Consumption']

norm_stress = scaler_tmp.fit_transform(df[stress_features])
df['Stress_Score'] = norm_stress.mean(axis=1)

upper_threshold = df['Stress_Score'].quantile(0.99)
lower_threshold = df['Stress_Score'].quantile(0.01)

df['UpperBoundary_Stress_Risk'] = (df['Stress_Score'] >= upper_threshold).astype(int)
df['LowerBoundary_Stress_Risk'] = (df['Stress_Score'] <= lower_threshold).astype(int)
df['Shutdown_Risk'] = ((df['UpperBoundary_Stress_Risk'] == 1) | (df['LowerBoundary_Stress_Risk'] == 1)).astype(int)

print(f'Upper boundary threshold (99th pct): {upper_threshold:.4f}')
print(f'Lower boundary threshold (1st pct): {lower_threshold:.4f}')
print()
print(df['Shutdown_Risk'].value_counts())
print(f'Shutdown-risk days: {df["Shutdown_Risk"].sum()} / {len(df)} ({df["Shutdown_Risk"].mean()*100:.1f}%)')

Upper boundary threshold (99th pct): 0.5700
Lower boundary threshold (1st pct): 0.2580

Shutdown_Risk
0    1354
1      28
Name: count, dtype: int64
Shutdown-risk days: 28 / 1382 (2.0%)


In [ ]:

fig = go.Figure()

fig.add_trace(go.Scatter(
    y=df['Stress_Score'], mode='lines', name='Stress Score', line=dict(color='black')
))

# fig.add_trace(go.Scatter(
#    x=shutdown_idx, y=df.loc[shutdown_idx, 'Stress_Score'],
#    mode='markers', name='Shutdown Risk', marker=dict(color='red', size=6, symbol='circle')
# ))

fig.add_hline(y=upper_threshold, line_dash='dash', line_color='red', annotation_text='Upper Threshold (99th pct)')
fig.add_hline(y=lower_threshold, line_dash='dash', line_color='blue', annotation_text='Lower Threshold (1st pct)')

fig.update_layout(title='Plant Stress Score', height=400)
fig.show()

In [ ]:
model_features = [
    'Shutdown_Risk',
    'Chemical Oxygen Demand',
    'Biological Oxygen Demand',
    'Ammonia',
    'Total Nitrogen',
    'Average Inflow',
    'Average Outflow',
    'Energy Consumption',
    'Average Temperature',
    'Average humidity',
    'Total rainfall',
]

df_model = df[model_features].copy()
df_model.dropna(inplace=True)

print(f'Modelling dataframe shape: {df_model.shape}')
df_model.head()

Modelling dataframe shape: (1382, 11)


,Shutdown_Risk,Chemical Oxygen Demand,Biological Oxygen Demand,Ammonia,Total Nitrogen,Average Inflow,Average Outflow,Energy Consumption,Average Temperature,Average humidity,Total rainfall
0,0,730.0,365.0,27.0,60.378,2.589,2.941,175.9,19.3,56.0,1.52
1,0,740.0,370.0,25.0,60.026,2.961,2.936,181.6,17.1,63.0,0.00
2,0,836.0,418.0,42.0,64.522,3.225,2.928,202.0,16.8,47.0,0.25
3,0,850.0,430.0,36.0,63.000,3.354,2.928,207.5,14.6,49.0,0.00
4,0,1016.0,508.0,46.0,65.590,3.794,2.917,202.8,13.4,65.0,0.00


In [ ]:
fig = px.imshow(
    df_model.corr(),
    title='Feature Correlations',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1
)
fig.show()

In [ ]:
def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
    n_vars = 1 if type(data) is list else data.shape[1]
    dff = pd.DataFrame(data)
    cols, names = [], []
    for i in range(n_in, 0, -1):
        cols.append(dff.shift(i))
        names += [f'var{j+1}(t-{i})' for j in range(n_vars)]
    for i in range(0, n_out):
        cols.append(dff.shift(-i))
        if i == 0:
            names += [f'var{j+1}(t)' for j in range(n_vars)]
        else:
            names += [f'var{j+1}(t+{i})' for j in range(n_vars)]
    agg = pd.concat(cols, axis=1)
    agg.columns = names
    if dropnan:
        agg.dropna(inplace=True)
    return agg


LOOKBACK = 10  # days of history
N_FEATURES = df_model.shape[1]  # includes the target

scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(df_model.values)

reframed = series_to_supervised(scaled, n_in=LOOKBACK, n_out=1, dropnan=True)

cols_to_drop = [c for c in reframed.columns if '(t)' in c and not c.startswith('var1(t)')]
reframed.drop(columns=cols_to_drop, inplace=True)

print(f'Reframed shape: {reframed.shape}')
print('Last columns (target should be last):', reframed.columns[-3:].tolist())

Reframed shape: (1372, 111)
Last columns (target should be last): ['var10(t-1)', 'var11(t-1)', 'var1(t)']


In [ ]:

values = reframed.values
n_train = int(len(values) * 0.25)

train = values[:n_train, :]
test  = values[n_train:, :]

train_X, train_y = train[:, :-1], train[:, -1]
test_X,  test_y  = test[:, :-1],  test[:, -1]


train_X = train_X.reshape((train_X.shape[0], LOOKBACK, N_FEATURES))
test_X  = test_X.reshape((test_X.shape[0],  LOOKBACK, N_FEATURES))

print(f'Train X: {train_X.shape} | Train y: {train_y.shape}')
print(f'Test  X: {test_X.shape}  | Test  y: {test_y.shape}')

Train X: (343, 10, 11) | Train y: (343,)
Test  X: (1029, 10, 11)  | Test  y: (1029,)


In [ ]:
model = Sequential([
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    Dense(1, activation='sigmoid')   # sigmoid → probability of Shutdown_Risk
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5)
]

classes = np.unique(train_y)
weights = compute_class_weight('balanced', classes=classes, y=train_y)
class_weight = dict(zip(classes, weights))

history = model.fit(
    train_X, train_y,
    epochs=100,
    batch_size=32,
    validation_data=(test_X, test_y),
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
    shuffle=False
)

Epoch 1/100


ValueError: Could not interpret loss identifier: tf.keras.losses.CategoricalFocalCrossentropy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Model Loss (Binary Crossentropy)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train Acc')
axes[1].plot(history.history['val_accuracy'], label='Val Acc')
axes[1].set_title('Model Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Predicted probabilities
y_prob = model.predict(test_X).flatten()

print(f'y_prob min: {y_prob.min():.4f}, max: {y_prob.max():.4f}, std: {y_prob.std():.4f}')
print(f'y_prob sample: {y_prob[:10]}')
print(f'test_y distribution:\n{pd.Series(test_y).value_counts()}')

# Binary predictions at 0.5 threshold
y_pred = (y_prob >= 0.5).astype(int)

print('Classification Report')
print('=====================')
print(classification_report(test_y.astype(int), y_pred, target_names=['Normal (0)', 'Shutdown Risk (1)']))

rmse = np.sqrt(mean_squared_error(test_y, y_prob))
print(f'RMSE (probability vs true label): {rmse:.4f}')

In [ ]:
# Confusion matrix
cm = confusion_matrix(test_y.astype(int), y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Shutdown Risk'],
            yticklabels=['Normal', 'Shutdown Risk'])
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
ax.set_title('Confusion Matrix — WWTP Shutdown Prediction')
plt.tight_layout()
plt.show()

In [ ]:
plot_n = min(len(test_y), 500)
t = np.arange(plot_n)

fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=test_y[:plot_n],
                         mode='lines+markers', name='Actual',
                         marker=dict(size=3), line=dict(color='black')))
fig.add_trace(go.Scatter(x=t, y=y_prob[:plot_n],
                         mode='lines', name='Predicted Probability',
                         line=dict(color='blue')))
fig.update_layout(
    title=f'Actual vs Predicted Shutdown Risk (first {plot_n} test days)',
    xaxis_title='Day Index',
    yaxis_title='Shutdown Risk',
    height=450
)
fig.show()